# ConvNeXt-CIFAR10 con Muon optimizer

Notebook completo:

- `data.py` — pipeline dati (CIFAR-10, augmentation, Mixup factory)
- `model.py` — ConvNeXt adattato per CIFAR (stem stride=1, kernel per-stage)
- `muon.py` — Muon optimizer (Newton-Schulz orthogonalization)
- `train.py` — training loop con supporto Muon+AdamW oppure AdamW solo

Per switchare tra AdamW e Muon+AdamW basta il flag `use_muon` in `fit()`.


## Setup

In [1]:
!pip install -q timm

## `data.py`

In [ ]:
%%writefile data.py
"""data.py — Data pipeline per ConvNeXt-CIFAR.

Espone:
    DEVICE                 torch.device auto-detected (cuda > mps > cpu)
    CIFAR_MEAN, CIFAR_STD  statistiche per-canale del training set
    NUM_CLASSES            10
    train_transform        crop + flip + RandAugment + normalize
    val_transform          solo ToTensor + normalize
    build_loaders(...)     loader completo per training (CIFAR-10 full)
    smoke_loaders(...)     loader minimal per smoke test su MPS/CPU
    build_mixup(...)       factory per timm Mixup
"""

import os
os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')

import numpy as np
import torch
import torchvision
from torch.utils.data import DataLoader, Subset
from torchvision import transforms
from torchvision.transforms import AutoAugment, AutoAugmentPolicy

# --- Constants ---------------------------------------------------------
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2470, 0.2435, 0.2616)
NUM_CLASSES = 10


# --- Device ------------------------------------------------------------
def get_device() -> torch.device:
    """cuda > mps > cpu, in quest'ordine di preferenza."""
    if torch.cuda.is_available():
        return torch.device('cuda')
    if torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')


DEVICE = get_device()


# --- Transforms --------------------------------------------------------
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=6, padding_mode='reflect'),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.TrivialAugmentWide(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.15), value='random'),
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])


# --- DataLoaders -------------------------------------------------------
def _loader_kwargs(num_workers):
    """Defaults sensati per (num_workers, pin_memory, persistent_workers)."""
    if num_workers is None:
        num_workers = {'cuda': 4, 'mps': 0, 'cpu': 2}[DEVICE.type]
    pin_memory = (DEVICE.type == 'cuda')
    persistent = (num_workers > 0)
    return num_workers, pin_memory, persistent


def build_loaders(data_root='./data', batch_size=256, num_workers=None):
    """Carica CIFAR-10 completo con augmentation pipeline."""
    num_workers, pin_memory, persistent = _loader_kwargs(num_workers)

    train_set = torchvision.datasets.CIFAR10(
        root=data_root, train=True,  download=True, transform=train_transform)
    val_set = torchvision.datasets.CIFAR10(
        root=data_root, train=False, download=True, transform=val_transform)

    train_loader = DataLoader(
        train_set, batch_size=batch_size, shuffle=True,
        num_workers=num_workers, pin_memory=pin_memory,
        drop_last=True, persistent_workers=persistent,
    )
    val_loader = DataLoader(
        val_set, batch_size=batch_size * 2, shuffle=False,
        num_workers=num_workers, pin_memory=pin_memory,
        persistent_workers=persistent,
    )
    return train_loader, val_loader


def smoke_loaders(data_root='./data', batch_size=64,
                  n_train=1000, n_val=200, seed=0):
    """Mini DataLoader per smoke test del training loop."""
    rng = np.random.default_rng(seed)
    full_train = torchvision.datasets.CIFAR10(
        root=data_root, train=True,  download=True, transform=train_transform)
    full_val = torchvision.datasets.CIFAR10(
        root=data_root, train=False, download=True, transform=val_transform)

    train_idx = rng.choice(len(full_train), n_train, replace=False)
    val_idx   = rng.choice(len(full_val),   n_val,   replace=False)

    train_loader = DataLoader(
        Subset(full_train, train_idx), batch_size=batch_size, shuffle=True,
        num_workers=0, pin_memory=False, drop_last=True,
    )
    val_loader = DataLoader(
        Subset(full_val, val_idx), batch_size=batch_size * 2, shuffle=False,
        num_workers=0, pin_memory=False,
    )
    return train_loader, val_loader


# --- Mixup / CutMix ----------------------------------------------------
def build_mixup(mixup_alpha=0.2, cutmix_alpha=1.0, prob=1.0,
                switch_prob=0.5, label_smoothing=0.1, num_classes=NUM_CLASSES):
    """Factory per timm.data.Mixup. Default tarati per budget 60 epoche."""
    from timm.data import Mixup
    return Mixup(
        mixup_alpha=mixup_alpha,
        cutmix_alpha=cutmix_alpha,
        prob=prob,
        switch_prob=switch_prob,
        mode='batch',
        label_smoothing=label_smoothing,
        num_classes=num_classes,
    )


Writing data.py


## `model.py`

In [3]:
%%writefile model.py
"""model.py — ConvNeXt adattato per CIFAR-10."""

import torch
import torch.nn as nn
import torch.nn.functional as F


class LayerNorm(nn.Module):
    """LayerNorm con supporto channels_last e channels_first."""
    def __init__(self, normalized_shape, eps=1e-6, data_format='channels_last'):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(normalized_shape))
        self.bias = nn.Parameter(torch.zeros(normalized_shape))
        self.eps = eps
        self.data_format = data_format
        self.normalized_shape = (normalized_shape,)

    def forward(self, x):
        if self.data_format == 'channels_last':
            return F.layer_norm(x, self.normalized_shape, self.weight, self.bias, self.eps)
        u = x.mean(1, keepdim=True)
        s = (x - u).pow(2).mean(1, keepdim=True)
        x = (x - u) / torch.sqrt(s + self.eps)
        return self.weight[:, None, None] * x + self.bias[:, None, None]


class DropPath(nn.Module):
    """Stochastic depth (per-sample)."""
    def __init__(self, drop_prob=0.):
        super().__init__()
        self.drop_prob = drop_prob

    def forward(self, x):
        if self.drop_prob == 0. or not self.training:
            return x
        keep_prob = 1 - self.drop_prob
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        mask = x.new_empty(shape).bernoulli_(keep_prob)
        return x.div(keep_prob) * mask

class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        # Il collo di bottiglia: riduce i canali e poi li ripristina
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), # Squeeze spaziale (1x1xC)
            nn.Flatten(),
            nn.Linear(channels, channels // reduction, bias=False),
            nn.GELU(),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()             # Excitation: mappa i pesi tra 0 e 1
        )

    def forward(self, x):
        # x shape: (B, C, H, W)
        b, c, _, _ = x.size()
        # Calcola i pesi e fa il reshape a (B, C, 1, 1) per il broadcasting
        w = self.fc(x).view(b, c, 1, 1)
        return x * w # Ricalibrazione dinamica dei canali

class ConvNeXtBlock(nn.Module):
    def __init__(self, dim, kernel_size=5, drop_path=0., layer_scale_init=1e-6, use_se=False):
        super().__init__()
        self.dwconv = nn.Conv2d(dim, dim, kernel_size=kernel_size,
                                padding=kernel_size // 2, groups=dim)
        self.norm = LayerNorm(dim, eps=1e-6, data_format='channels_last')
        self.pwconv1 = nn.Linear(dim, 4 * dim)
        self.act = nn.GELU()
        self.pwconv2 = nn.Linear(4 * dim, dim)
        
        # Iniezione condizionale dell'SE Block (lavora in data_format='channels_first')
        self.se = SEBlock(dim, reduction=16) if use_se else nn.Identity()
        
        if layer_scale_init > 0:
            self.gamma = nn.Parameter(layer_scale_init * torch.ones(dim))
        else:
            self.register_parameter('gamma', None)
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()

    def forward(self, x):
        shortcut = x
        x = self.dwconv(x)
        x = x.permute(0, 2, 3, 1)
        x = self.norm(x)
        x = self.pwconv1(x)
        x = self.act(x)
        x = self.pwconv2(x)
        if self.gamma is not None:
            x = self.gamma * x
        x = x.permute(0, 3, 1, 2)
        
        # Applichiamo l'attenzione SE subito prima del DropPath residuo
        x = self.se(x) 
        
        return shortcut + self.drop_path(x)


class ConvNeXt(nn.Module):
    """ConvNeXt scalato per CIFAR-10.

    kernel_size accetta int (uniforme su tutti gli stage) o tuple di lunghezza 4
    (uno per stage). Es: kernel_size=(7, 5, 3, 3) per kernel decrescente.
    """
    def __init__(self, in_channel=3, num_classes=10,
                 depths=(2, 2, 6, 2), dims=(64, 128, 256, 512),
                 kernel_size=5, drop_path_rate=0.1,
                 layer_scale_init=1e-6, stem_stride=1):
        super().__init__()

        if isinstance(kernel_size, int):
            kernel_sizes = (kernel_size,) * 4
        else:
            kernel_sizes = tuple(kernel_size)
            assert len(kernel_sizes) == 4, (
                f"kernel_size deve avere lunghezza 4 (uno per stage), "
                f"ricevuto len={len(kernel_sizes)}"
            )

        # Stem + 3 downsample
        self.downsample_layers = nn.ModuleList()
        self.downsample_layers.append(nn.Sequential(
            nn.Conv2d(in_channel, dims[0], kernel_size=3, stride=stem_stride, padding=1),
            LayerNorm(dims[0], eps=1e-6, data_format='channels_first')
        ))
        for i in range(3):
            self.downsample_layers.append(nn.Sequential(
                LayerNorm(dims[i], eps=1e-6, data_format='channels_first'),
                nn.Conv2d(dims[i], dims[i + 1], kernel_size=2, stride=2)
            ))

        # 4 stage di ConvNeXt block
        self.stages = nn.ModuleList()
        dp_rates = torch.linspace(0, drop_path_rate, sum(depths)).tolist()
        cur = 0
        for i in range(4):
            # use_se sarà True SOLO per lo stage i == 2 (ovvero lo Stage 3 da 9 blocchi)
            use_se_stage = (i == 2) 
            
            self.stages.append(nn.Sequential(*[
                ConvNeXtBlock(dim=dims[i], kernel_size=kernel_sizes[i],
                              drop_path=dp_rates[cur + j],
                              layer_scale_init=layer_scale_init,
                              use_se=use_se_stage) 
                for j in range(depths[i])
            ]))
            cur += depths[i]
        self.norm = nn.LayerNorm(dims[-1], eps=1e-6)
        self.head = nn.Linear(dims[-1], num_classes)

        self.apply(self.init_weights)

    def init_weights(self, m):
        if isinstance(m, (nn.Linear, nn.Conv2d)):
            nn.init.trunc_normal_(m.weight, std=.02)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)

    def forward_features(self, x):
        for downsample_layer, stage in zip(self.downsample_layers, self.stages):
            x = downsample_layer(x)
            x = stage(x)
        return self.norm(x.mean([-2, -1]))

    def forward(self, x):
        return self.head(self.forward_features(x))


Writing model.py


## `muon.py`

In [4]:
%%writefile muon.py
"""muon.py — Muon optimizer (Keller Jordan et al.).

MomentUm Orthogonalized by Newton-Schulz: ortogonalizza l'update via NS
iteration prima di applicarlo. Solo per parametri 2D/4D (Linear, Conv).
I parametri 1D (bias, norm, gamma) devono usare AdamW separato.

Riferimento: https://github.com/KellerJordan/Muon
"""

import torch


@torch.no_grad()
def zeropower_via_newtonschulz5(G, steps=5, eps=1e-7):
    """Newton-Schulz iteration: approssima G -> G * (G^T G)^(-1/2).
    Lavora in bfloat16 per stabilita' numerica e velocita'.
    """
    assert G.ndim == 2
    a, b, c = 3.4445, -4.7750, 2.0315
    X = G.bfloat16()
    if G.size(0) > G.size(1):
        X = X.T
    X = X / (X.norm() + eps)
    for _ in range(steps):
        A = X @ X.T
        B = b * A + c * (A @ A)
        X = a * X + B @ X
    if G.size(0) > G.size(1):
        X = X.T
    return X.to(G.dtype)


class Muon(torch.optim.Optimizer):
    """Muon per parametri 2D (Linear) e 4D (Conv).

    Update per parametro p:
        v_t = momentum * v_{t-1} + g_t
        g~  = g_t + momentum * v_t        (Nesterov) o solo v_t
        u   = newton_schulz(reshape_2D(g~))   # orthogonalized
        p   = (1 - lr*wd) * p - lr * sqrt(max(1, fan_out/fan_in)) * u
    """
    def __init__(self, params, lr=0.02, weight_decay=0.0,
                 momentum=0.95, nesterov=True, ns_steps=5):
        defaults = dict(lr=lr, weight_decay=weight_decay,
                        momentum=momentum, nesterov=nesterov, ns_steps=ns_steps)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self):
        for group in self.param_groups:
            lr = group['lr']
            wd = group['weight_decay']
            momentum = group['momentum']
            nesterov = group['nesterov']
            ns_steps = group['ns_steps']

            for p in group['params']:
                if p.grad is None:
                    continue
                grad = p.grad
                state = self.state[p]

                if 'momentum_buffer' not in state:
                    state['momentum_buffer'] = torch.zeros_like(p)
                buf = state['momentum_buffer']

                # SGD momentum
                buf.mul_(momentum).add_(grad)
                update = grad.add(buf, alpha=momentum) if nesterov else buf

                # Reshape 4D conv -> 2D matrix
                orig_shape = update.shape
                if update.ndim == 4:
                    update = update.view(update.size(0), -1)

                # Newton-Schulz orthogonalization
                update = zeropower_via_newtonschulz5(update, steps=ns_steps)

                if len(orig_shape) == 4:
                    update = update.view(orig_shape)

                # Aspect-ratio scaling
                fan_out = p.size(0)
                fan_in = p.numel() // fan_out
                scale = max(1.0, fan_out / max(fan_in, 1)) ** 0.5

                # Decoupled weight decay + apply update
                if wd != 0:
                    p.mul_(1.0 - lr * wd)
                p.add_(update, alpha=-lr * scale)


Writing muon.py


## `train.py`

In [ ]:
%%writefile train.py
"""train.py - Training loop per ConvNeXt-CIFAR.

Supporta due regimi di optimizer:
    use_muon=False  -> AdamW solo (default ConvNeXt recipe)
    use_muon=True   -> Muon su 2D/4D weights + AdamW su 1D + head

Componenti:
    fit(...)                        main training loop
    train_one_epoch(...)            singolo epoch (forward + backward + step)
    evaluate(...)                   validation con accuracy (no TTA)
    evaluate_tta(...)               validation con horizontal-flip TTA
    param_groups(...)               split per AdamW solo
    param_groups_muon_adamw(...)    split per Muon+AdamW
    cosine_warmup_lr(...)           LR schedule lineare + cosine
"""

import math
import time

import torch
import torch.nn as nn

from data import DEVICE


# --- Parameter groups -------------------------------------------------
def param_groups(model, base_lr, weight_decay):
    """AdamW solo: groups con WD separato (no WD su bias/LN/gamma)."""
    decay, no_decay, gamma_params = [], [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if 'gamma' in name:
            gamma_params.append(p)
        elif p.ndim <= 1 or 'norm' in name or 'bias' in name:
            no_decay.append(p)
        else:
            decay.append(p)
    return [
        {'params': decay,        'lr': base_lr,       'weight_decay': weight_decay},
        {'params': no_decay,     'lr': base_lr,       'weight_decay': 0.0},
        {'params': gamma_params, 'lr': base_lr , 'weight_decay': 0.0,},
    ]


def param_groups_muon_adamw(model, muon_wd=0.0, adamw_wd=0.02):
    """Split Muon+AdamW: 2D/4D hidden -> Muon, 1D + head -> AdamW.

    Convenzione (sul tuo ConvNeXt):
        Muon:   pwconv1.weight, pwconv2.weight, dwconv.weight,
                stem conv weight, downsample conv weights
        AdamW:  tutti i bias, LayerNorm weight/bias, LayerScale gamma,
                head.weight, head.bias
    """
    muon_params, adamw_decay, adamw_no_decay, gamma_params = [], [], [], []

    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        # Output head -> AdamW
        if name.startswith('head.'):
            (adamw_no_decay if 'bias' in name else adamw_decay).append(p)
            continue
        # LayerScale gamma -> AdamW con LR ridotto
        if 'gamma' in name:
            gamma_params.append(p)
            continue
        # 1D params (bias, norm) -> AdamW no decay
        if p.ndim <= 1 or 'norm' in name or 'bias' in name:
            adamw_no_decay.append(p)
            continue
        # 2D+ weights -> Muon
        muon_params.append(p)

    muon_groups = [{'params': muon_params, 'weight_decay': muon_wd}]
    adamw_groups = [
        {'params': adamw_decay,    'weight_decay': adamw_wd},
        {'params': adamw_no_decay, 'weight_decay': 0.0},
        {'params': gamma_params,   'weight_decay': 0.0},
    ]
    return muon_groups, adamw_groups


# --- LR schedule ------------------------------------------------------
def cosine_warmup_lr(step, total_steps, warmup_steps, base_lr, min_lr=1e-6):
    """LR per lo step corrente. Lineare durante warmup, poi cosine."""
    if step < warmup_steps:
        return base_lr * (step + 1) / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return min_lr + 0.5 * (base_lr - min_lr) * (1 + math.cos(math.pi * progress))


def set_lr(optimizer, lr):
    for g in optimizer.param_groups:
        g['lr'] = lr * g.get('lr_scale', 1.0)


# --- Training step ----------------------------------------------------
def train_one_epoch(model, loader, optimizers, criterion, mixup_fn,
                    schedule_fn, step_counter,
                    ema=None, scaler=None, grad_clip=1.0):
    """optimizers: lista di (optimizer, base_lr) tuple.

    Per AdamW-solo: optimizers = [(adamw, base_lr)]
    Per Muon+AdamW: optimizers = [(adamw, adamw_base_lr), (muon, muon_base_lr)]
    """
    model.train()
    total_loss = 0.0
    n = 0
    use_amp = scaler is not None

    for x, y in loader:
        # LR per ogni optimizer (ciascuno con il proprio base_lr)
        for opt, base_lr in optimizers:
            set_lr(opt, schedule_fn(step_counter['v'], base_lr))

        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        if mixup_fn is not None:
            x, y = mixup_fn(x, y)

        for opt, _ in optimizers:
            opt.zero_grad(set_to_none=True)

        if use_amp:
            with torch.amp.autocast('cuda', dtype=torch.float16):
                logits = model(x)
                loss = criterion(logits, y)
            scaler.scale(loss).backward()
            for opt, _ in optimizers:
                scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            for opt, _ in optimizers:
                scaler.step(opt)
            scaler.update()
        else:
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            for opt, _ in optimizers:
                opt.step()

        if ema is not None:
            ema.update(model._orig_mod if hasattr(model, '_orig_mod') else model)

        step_counter['v'] += 1
        bs = x.size(0)
        total_loss += loss.item() * bs
        n += bs

    return total_loss / max(1, n)


# --- Validation -------------------------------------------------------
@torch.no_grad()
def evaluate(model, loader):
    """Accuracy + loss CE standard, no TTA."""
    model.eval()
    crit = nn.CrossEntropyLoss()
    total_loss = 0.0
    n_correct = 0
    n_total = 0
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        logits = model(x)
        loss = crit(logits, y)
        bs = x.size(0)
        total_loss += loss.item() * bs
        n_correct += (logits.argmax(dim=1) == y).sum().item()
        n_total += bs
    return total_loss / n_total, n_correct / n_total


@torch.no_grad()
def evaluate_tta(model, loader):
    """Accuracy + loss con horizontal-flip TTA (media softmax)."""
    model.eval()
    total_loss = 0.0
    n_correct = 0
    n_total = 0
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        probs = (model(x).softmax(dim=1) + model(x.flip(-1)).softmax(dim=1)) / 2.0
        loss = nn.functional.nll_loss(probs.clamp(min=1e-12).log(), y)
        bs = x.size(0)
        total_loss += loss.item() * bs
        n_correct  += (probs.argmax(dim=1) == y).sum().item()
        n_total    += bs
    return total_loss / n_total, n_correct / n_total

@torch.no_grad()
def evaluate_multicrop_tta(model, loader, padding=4):
    """Multi-crop + horizontal flip TTA.
    
    Per ogni immagine 32x32:
      1. Pad reflect a 40x40 (matching la pipeline di training)
      2. Estrai 5 crop 32x32 (4 angoli + centro)
      3. Applica horizontal flip a ciascuno -> 10 viste totali
      4. Media softmax su tutte le viste
    
    Atteso: +0.1-0.2pp rispetto al solo flip-TTA, ~5x piu' lento in eval.
    """
    model.eval()
    correct = 0
    total = 0
    total_loss = 0.0
    
    crop_offset = padding * 2   # offset per i 4 corner crop
    center_start = padding      # offset per il center crop
    
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        bs = x.size(0)
        
        # Pad reflect (stesso padding del training transform)
        x_pad = nn.functional.pad(x, [padding] * 4, mode='reflect')
        # x_pad shape: (bs, 3, 32+2*padding, 32+2*padding)
        
        # Estrai 5 crop di 32x32
        crops = torch.stack([
            x_pad[:, :, :32, :32],                          # top-left
            x_pad[:, :, :32, crop_offset:crop_offset+32],   # top-right
            x_pad[:, :, crop_offset:crop_offset+32, :32],   # bottom-left
            x_pad[:, :, crop_offset:crop_offset+32, crop_offset:crop_offset+32],  # bottom-right
            x_pad[:, :, center_start:center_start+32, center_start:center_start+32],  # center
        ], dim=0)  # (5, bs, 3, 32, 32)
        
        # Reshape per forward batch: (5*bs, 3, 32, 32)
        crops_flat = crops.view(5 * bs, 3, 32, 32)
        
        # Forward su originale + horizontal flip
        probs = model(crops_flat).softmax(dim=1)
        probs = probs + model(crops_flat.flip(-1)).softmax(dim=1)

        # Media: (5, bs, 10) -> mean su crop axis -> div 2 per le due flip-views
        probs = probs.view(5, bs, 10).mean(dim=0) / 2.0
        
        # NLL loss e accuracy
        loss = nn.functional.nll_loss(probs.clamp(min=1e-12).log(), y)
        total_loss += loss.item() * bs
        correct += (probs.argmax(dim=1) == y).sum().item()
        total += bs
    
    return total_loss / total, correct / total
# --- Main fit ---------------------------------------------------------
def fit(model, train_loader, val_loader,
        epochs=60, base_lr=4e-3, weight_decay=0.02,
        warmup_epochs=5, grad_clip=1.0,
        use_mixup=True, use_ema=True, use_amp=True,
        use_muon=False, muon_lr=0.01, muon_momentum=0.95, muon_wd=0.0,
        mixup_alpha=0.2, cutmix_alpha=1.0,
        label_smoothing=0.1, ema_decay=0.999, use_compile=False,
        save_path="best.pt"):
    """Training loop completo.

    Args principali:
        epochs:           numero totale di epoch
        base_lr:          peak LR per AdamW
        weight_decay:     WD per i weight di Linear/Conv in AdamW (no su bias/LN/gamma)
        warmup_epochs:    durata warmup lineare
        use_muon:         True per Muon (2D/4D) + AdamW (1D/head)
        muon_lr:          peak LR per Muon (tipicamente 2-10x AdamW)
        muon_momentum:    momentum SGD-base per Muon (default 0.95)
        muon_wd:          WD applicato dentro Muon (default 0, raramente serve)
        mixup_alpha:      Beta(alpha, alpha) per Mixup. 0.2 leggero per budget corti
        cutmix_alpha:     idem per CutMix. 1.0 = uniform mixing
        ema_decay:        decay EMA. 0.999 per 60 ep, 0.9995-0.9999 per 100+
    """
    # Criterion + mixup
    if use_mixup:
        from timm.loss import SoftTargetCrossEntropy
        from data import build_mixup
        mixup_fn = build_mixup(
            mixup_alpha=mixup_alpha,
            cutmix_alpha=cutmix_alpha,
            prob=1.0,
            switch_prob=0.5,
            label_smoothing=label_smoothing,
        )
        criterion = SoftTargetCrossEntropy()
    else:
        mixup_fn = None
        criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)

    # Optimizer(s)
    if use_muon:
        from muon import Muon
        muon_groups, adamw_groups = param_groups_muon_adamw(
            model, muon_wd=muon_wd, adamw_wd=weight_decay
        )
        muon_opt  = Muon(muon_groups, lr=muon_lr, momentum=muon_momentum)
        adamw_opt = torch.optim.AdamW(adamw_groups, lr=base_lr,
                                       betas=(0.9, 0.999), eps=1e-8)
        optimizers = [(adamw_opt, base_lr), (muon_opt, muon_lr)]
        opt_str = (f"Muon(lr={muon_lr}, mom={muon_momentum}, wd={muon_wd}) + "
                   f"AdamW(lr={base_lr:.0e}, wd={weight_decay})")
    else:
        adamw_opt = torch.optim.AdamW(
            param_groups(model, base_lr, weight_decay),
            betas=(0.9, 0.999), eps=1e-8,
        )
        optimizers = [(adamw_opt, base_lr)]
        opt_str = f"AdamW(lr={base_lr:.0e}, wd={weight_decay})"

    # EMA
    ema = None
    if use_ema:
        from timm.utils import ModelEmaV2
        ema = ModelEmaV2(model, decay=ema_decay, device=DEVICE)

    if use_compile:
        model = torch.compile(model, mode="reduce-overhead")

    # AMP scaler 
    scaler = None
    if use_amp and DEVICE.type == 'cuda':
        scaler = torch.amp.GradScaler('cuda')

    # Schedule (per-optimizer, ciascuno con il proprio base_lr)
    steps_per_epoch = len(train_loader)
    total_steps = epochs * steps_per_epoch
    warmup_steps = warmup_epochs * steps_per_epoch
    schedule_fn = lambda s, base: cosine_warmup_lr(s, total_steps, warmup_steps, base)
    step_counter = {'v': 0}

    # Header
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"Device={DEVICE.type}  amp={scaler is not None}  "
          f"ema={ema is not None}  mixup={mixup_fn is not None}  muon={use_muon}")
    print(f"Params={n_params:.2f}M  epochs={epochs}  "
          f"steps/ep={steps_per_epoch}  total={total_steps}  warmup={warmup_steps}")
    print(f"Optimizer: {opt_str}")
    print(f"ls={label_smoothing}  clip={grad_clip}  ema_decay={ema_decay}")
    print('-' * 100)

    best_acc = 0.0
    best_kind = "model"

    for epoch in range(epochs):
        t0 = time.time()
        train_loss = train_one_epoch(
            model, train_loader, optimizers, criterion, mixup_fn,
            schedule_fn, step_counter,
            ema=ema, scaler=scaler, grad_clip=grad_clip,
        )

        val_loss, val_acc = evaluate(model, val_loader)
        ema_loss, ema_acc = None, None
        if ema is not None:
            ema_loss, ema_acc = evaluate(ema.module, val_loader)

        current_lr = optimizers[0][0].param_groups[0]['lr']
        dt = time.time() - t0

        # Resolve underlying model (strip torch.compile wrapper se presente)
        base_model = model._orig_mod if hasattr(model, '_orig_mod') else model

        # Pick best between live model and EMA
        if ema_acc is not None and ema_acc >= val_acc:
            current_best_acc = ema_acc
            current_best_loss = ema_loss
            current_state = ema.module.state_dict()
            current_kind = "ema"
        else:
            current_best_acc = val_acc
            current_best_loss = val_loss
            current_state = base_model.state_dict()        
            current_kind = "model"

        if current_best_acc > best_acc:
            best_acc = current_best_acc
            best_kind = current_kind
            torch.save({
                "epoch": epoch + 1,
                "model_state": current_state,
                "best_acc": best_acc,
                "val_acc": val_acc,
                "val_loss": val_loss,
                "ema_acc": ema_acc,
                "ema_loss": ema_loss,
                "kind": current_kind,
                "config": {
                    "epochs": epochs,
                    "base_lr": base_lr,
                    "weight_decay": weight_decay,
                    "warmup_epochs": warmup_epochs,
                    "grad_clip": grad_clip,
                    "use_mixup": use_mixup,
                    "use_ema": use_ema,
                    "use_muon": use_muon,
                    "muon_lr": muon_lr if use_muon else None,
                    "muon_momentum": muon_momentum if use_muon else None,
                    "muon_wd": muon_wd if use_muon else None,
                    "mixup_alpha": mixup_alpha,
                    "cutmix_alpha": cutmix_alpha,
                    "label_smoothing": label_smoothing,
                    "ema_decay": ema_decay,
                }
            }, save_path)
            print(f"  -> saved best checkpoint: {save_path} "
                  f"({current_kind}, acc={best_acc:.4f}, loss={current_best_loss:.4f})")

        # Backup periodico ogni 50 epoche (per resume in caso di timeout Kaggle)
        if (epoch + 1) % 50 == 0:
            torch.save({
                "epoch": epoch + 1,
                "model_state": base_model.state_dict(),
                "ema_state": ema.module.state_dict() if ema is not None else None,
                "best_acc": best_acc,
            }, save_path.replace(".pt", "_last.pt"))

    print('-' * 100)
    print(f"Best validation accuracy: {best_acc:.4f}")
    return best_acc


# --- Self test --------------------------------------------------------
if __name__ == '__main__':
    from data import build_loaders
    from model import ConvNeXt

    torch.backends.cudnn.benchmark = True

    train_loader, val_loader = build_loaders(
        data_root="/kaggle/working/data",
        batch_size=256,
        num_workers=4,
    )

    model = ConvNeXt(
        depths=(3, 3, 9, 3),
        dims=(64, 128, 256, 512),
        kernel_size=(7, 5, 3, 3),
        drop_path_rate=0.15,
        layer_scale_init=1e-6,
    ).to(DEVICE)
    
    fit(
        model, train_loader, val_loader,
        epochs=600,
        warmup_epochs=30,
        base_lr=1e-3,
        muon_lr=0.02,
        muon_momentum=0.95,
        muon_wd=0.02,
        weight_decay=0.0,
        grad_clip=1.0,
        use_mixup=True,
        mixup_alpha=0.2,
        cutmix_alpha=1.0,
        use_ema=True,
        use_amp=True,
        use_muon=True,
        use_compile=True,  
        label_smoothing=0.1,
        ema_decay=0.9997,
        save_path="/kaggle/working/best_600ep_compile.pt",
    )
        


Writing train.py


## Sanity check (param split + kernel per stage)

## Run training

In [6]:
!python -u train.py

100%|████████████████████████████████████████| 170M/170M [00:03<00:00, 48.6MB/s]
Device=cuda  amp=True  ema=True  mixup=True  muon=True
Params=12.37M  epochs=600  steps/ep=195  total=117000  warmup=5850
Optimizer: Muon(lr=0.02, mom=0.95, wd=0.02) + AdamW(lr=1e-03, wd=0.0)
ls=0.1  clip=1.0  ema_decay=0.9997
----------------------------------------------------------------------------------------------------
W0522 15:15:37.510000 49 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode
  -> saved best checkpoint: /kaggle/working/best_600ep_compile.pt (model, acc=0.2839, loss=2.0298)
  -> saved best checkpoint: /kaggle/working/best_600ep_compile.pt (model, acc=0.3079, loss=1.9828)
  -> saved best checkpoint: /kaggle/working/best_600ep_compile.pt (model, acc=0.3606, loss=1.8253)
  -> saved best checkpoint: /kaggle/working/best_600ep_compile.pt (model, acc=0.4146, loss=1.6799)
  -> saved best checkpoint: /kaggle/working/best_600ep_compile.pt (model, acc=0.4768, lo

## Evaluation con TTA

In [7]:
import torch
import importlib
import data, train
for m in [data, train]:
    importlib.reload(m)

from data import build_loaders, DEVICE
from model import ConvNeXt
from train import evaluate, evaluate_tta, evaluate_multicrop_tta

_, val_loader = build_loaders(
    data_root="/kaggle/working/data",
    batch_size=128,   
)

def make_model(depths=(3, 3, 9, 3), drop_path=0.15):
    return ConvNeXt(
        depths=depths,
        dims=(64, 128, 256, 512),
        kernel_size=(7, 5, 3, 3),
        drop_path_rate=drop_path,
        layer_scale_init=1e-6,
    ).to(DEVICE)

checkpoints = [
    ("Custom ConvNeXt", "/kaggle/working/best_600ep_compile.pt",    (3,3,9,3), 0.15),
]

print(f"{'Run':<22s} {'no TTA':>10s} {'flip TTA':>10s} {'5-crop TTA':>12s} {'Δ vs flip':>10s}")
print("-" * 70)

for name, path, depths, dp in checkpoints:
    model = make_model(depths=depths, drop_path=dp)
    ckpt = torch.load(path, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state"])
    
    _, acc_plain = evaluate(model, val_loader)
    _, acc_flip  = evaluate_tta(model, val_loader)
    _, acc_5crop = evaluate_multicrop_tta(model, val_loader)
    
    delta = (acc_5crop - acc_flip) * 100
    print(f"{name:<22s} {acc_plain*100:>9.2f}% {acc_flip*100:>9.2f}% "
          f"{acc_5crop*100:>11.2f}% {delta:>+9.2f}pp")


Run                        no TTA   flip TTA   5-crop TTA  Δ vs flip
----------------------------------------------------------------------
Custom ConvNeXt            98.10%     98.08%       98.00%     -0.08pp
